In [2]:
# Importation des modules
# Import bibliothèque de manipulation de dataframe
import pandas as pd

# Import des bibliothèques de viz
import matplotlib.pyplot as plt
import seaborn as sns

# Import split data
from sklearn.model_selection import train_test_split

# Import modèles de ML Supervisé Régression
from sklearn.linear_model import LinearRegression

# Import modèles de ML Supervisé Classification
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Import modèle de ML NON Supervisé
from sklearn.neighbors import NearestNeighbors

# Import des métriques
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# Import outil standardisation de la donnée
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

# Import pipeline
from sklearn.pipeline import Pipeline

from sklearn.base import BaseEstimator, TransformerMixin

# Gestion des warnings
import warnings

In [3]:
# Custom transformer for MultiLabelBinarizer
class MultiLabelBinarizerPipelineFriendly(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.mlb = MultiLabelBinarizer()

    def fit(self, X, y=None):
        self.mlb.fit(X)
        return self

    def transform(self, X):
        return self.mlb.transform(X)

    def get_feature_names_out(self, input_features=None):
        return self.mlb.classes_

In [4]:
# Récuperation du df
df = pd.read_csv('../ressources/df_v3.csv', sep=';', encoding='utf-8')
df

,frenchTitle,genres,averageRating,numVotes,actor1,actor2,actor3,decade
0,Les vampires,"Crime, Thriller, Drama, Action, Adventure",7.3,5696,Musidora,Édouard Mathé,Marcel Lévesque,1910
1,Judex,"Crime, Thriller, Drama, Mystery, Adventure",7.2,1194,René Cresté,Musidora,René Poyen,1910
2,J'accuse,"Horror, Drama, Romance, History, War",7.7,2248,Romuald Joubé,Maxime Desjardins,Séverin-Mars,1910
3,Coeur fidèle,"Romance, Drama",7.4,1543,Léon Mathot,Gina Manès,Edmond Van Daële,1920
4,La rose du rail,Drama,7.5,2686,Gabriel de Gravone,Pierre Magnier,Georges Térof,1920
...,...,...,...,...,...,...,...,...
12905,Herself,Drama,7.0,5124,Molly McCann,Clare Dunne,Ruby Rose O'Hara,2020
12906,Enemy Lines,"War, Drama, Action",4.6,2045,Ed Westwick,John Hannah,Tom Wisdom,2020
12907,Le lion,Comedy,5.5,1515,Dany Boon,Philippe Katerine,Anne Serra,2020
12908,Safeguard,"Crime, Thriller, Action, Adventure",3.6,263,Patrick Gallagher,Akie Kotabe,Sean Cronin,2020


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Preprocessor
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [5]:
X = df.drop(columns=['frenchTitle'])
films_non_standardise = X.iloc[:3]

In [ ]:
# Preprocessor pour standardiser les colonnes numériques
preprocessor = ColumnTransformer(
    transformers=[
        ('actors', OneHotEncoder(), ['actor1', 'actor2', 'actor3']),
        ('genres', MultiLabelBinarizerPipelineFriendly(), 'genres'),
        ('encoder', OrdinalEncoder(), ['decade']),
        ('skip', 'passthrough', ['averageRating', 'numVotes']),
    ]
)


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Pipeline
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
# Création du pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('knn', NearestNeighbors(n_neighbors=5))
    ]
)

pipeline.fit(X)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('encoder', OrdinalEncoder(),
                                                  ['decade']),
                                                 ('genres',
                                                  MultiLabelBinarizerPipelineFriendly(),
                                                  'genres'),
                                                 ('actors', OneHotEncoder(),
                                                  ['actor1', 'actor2',
                                                   'actor3']),
                                                 ('skip', 'passthrough',
                                                  ['averageRating',
                                                   'numVotes'])])),
                ('knn', NearestNeighbors())])

In [9]:
point_test = X.iloc[:3]
# Prédiction des voisins les plus proches
X_test_transformed = pipeline.named_steps['preprocessor'].transform(point_test)

distances, indices = pipeline.named_steps['knn'].kneighbors(X_test_transformed)
# Affichage des indices des voisins les plus proches
print("Indices des voisins les plus proches :", indices)
# Affichage des distances des voisins les plus proches
print("Distances des voisins les plus proches :", distances)


Indices des voisins les plus proches : [[   0 4173 8035 8573 2564]
 [   1  154  297  720 1304]
 [   2   46   64 1226  199]]
Distances des voisins les plus proches : [[ 0.         10.02796091 11.00181803 11.07429456 11.37936729]
 [ 0.          5.4         5.7280014   6.93974063  6.94622199]
 [ 0.          5.74456265  6.86731389  7.83836718  8.84081444]]


In [10]:
print(f"\nRecherche des 5 plus proches voisins pour 3 films (basé sur les features standardisées de X_class):")
for i in range(len(point_test)):
    # Récupérer l'index original du point exemple dans le DataFrame df
    original_index = point_test.index[i]
    print(f"\n--- Voisinage pour Film Exemple {i+1} (Index original: {original_index}) ---")
    print(f"  Note moyenne: {df.loc[original_index, 'averageRating']}")

    # Les 'indices' renvoyés par kneighbors sont les positions (0, 1, 2...) dans X utilisé lors du fit
    print(f"  Indices des voisins dans X: {indices[i]}")
    print(f"  Distances euclidiennes aux voisins: {distances[i]}")

    # Pour afficher les informations des voisins, il faut retrouver leurs index originaux dans df.
    neighbor_original_indices = X.iloc[indices[i]].index
    print(f"  Index originaux des voisins dans le DataFrame: {list(neighbor_original_indices)}")

    print("  Infos sur les voisins trouvés (depuis df original):")
    neighbor_info = df.loc[neighbor_original_indices][['averageRating', 'numVotes', 'decade']]
    print(neighbor_info)


Recherche des 5 plus proches voisins pour 3 films (basé sur les features standardisées de X_class):

--- Voisinage pour Film Exemple 1 (Index original: 0) ---
  Note moyenne: 7.3
  Indices des voisins dans X: [   0 4173 8035 8573 2564]
  Distances euclidiennes aux voisins: [ 0.         10.02796091 11.00181803 11.07429456 11.37936729]
  Index originaux des voisins dans le DataFrame: [0, 4173, 8035, 8573, 2564]
  Infos sur les voisins trouvés (depuis df original):
      averageRating  numVotes  decade
0               7.3      5696    1910
4173            5.7      5698    2000
8035            7.1      5696    2010
8573            6.5      5698    2010
2564            6.6      5689    1990

--- Voisinage pour Film Exemple 2 (Index original: 1) ---
  Note moyenne: 7.2
  Indices des voisins dans X: [   1  154  297  720 1304]
  Distances euclidiennes aux voisins: [0.         5.4        5.7280014  6.93974063 6.94622199]
  Index originaux des voisins dans le DataFrame: [1, 154, 297, 720, 1304]

In [14]:
titres = ['Avengers', "Harry Potter à l'école des sorciers", 'Avatar']  # ou d’autres

for titre in titres:
    film_cible = df[df['frenchTitle'].str.contains(titre, case=False, na=False)]
    if film_cible.empty:
        print(f"Film '{titre}' non trouvé.")
        continue

    idx_film = film_cible.index[0]
    film_non_standardise = df.drop(columns=['frenchTitle']).loc[[idx_film]]
    film_transforme = pipeline.named_steps['preprocessor'].transform(film_non_standardise)
    distances, indices = pipeline.named_steps['knn'].kneighbors(film_transforme)

    print(f"\n🎬 Film : {df.loc[idx_film, 'frenchTitle']} (Index: {idx_film})")
    print(f"  Note moyenne : {df.loc[idx_film, 'averageRating']}")
    neighbor_original_indices = X.iloc[indices[0]].index
    neighbor_info = df.loc[neighbor_original_indices][['frenchTitle', 'averageRating', 'numVotes', 'decade', 'genres', 'actor1', 'actor2', 'actor3']]
    print("  Voisins :")
    display(neighbor_info)

Film 'Avengers' non trouvé.

🎬 Film : Harry Potter à l'école des sorciers (Index: 3705)
  Note moyenne : 7.7
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actor1,actor2,actor3
3705,Harry Potter à l'école des sorciers,7.7,905446,2000,"Fantasy, Adventure, Family",Daniel Radcliffe,Rupert Grint,Emma Watson
12068,Once Upon a Time in... Hollywood,7.6,911221,2010,"Thriller, Drama, Comedy",Leonardo DiCaprio,Brad Pitt,Margot Robbie
989,Orange mécanique,8.2,912550,1970,"Crime, Drama, Sci-Fi, ScienceFiction",Malcolm McDowell,Patrick Magee,Michael Bates
5908,Slumdog Millionaire,8.0,895619,2000,"Crime, Romance, Drama",Dev Patel,Freida Pinto,Saurabh Shukla
7807,Oppenheimer,8.3,895304,2020,"History, Drama, Biography",Cillian Murphy,Emily Blunt,Matt Damon



🎬 Film : Avatar (Index: 5465)
  Note moyenne : 7.9
  Voisins :


,frenchTitle,averageRating,numVotes,decade,genres,actor1,actor2,actor3
5465,Avatar,7.9,1432326,2000,"ScienceFiction, Fantasy, Action, Adventure",Sam Worthington,Zoe Saldaña,Sigourney Weaver
5367,Le Prestige,8.5,1517028,2000,"Mystery, Drama, Sci-Fi, ScienceFiction",Christian Bale,Hugh Jackman,Scarlett Johansson
2405,Léon,8.5,1304475,1990,"Crime, Drama, Action",Jean Reno,Gary Oldman,Natalie Portman
2010,Terminator 2 : Le Jugement dernier,8.6,1234425,1990,"Thriller, Action, Adventure, Sci-Fi, ScienceFi...",Arnold Schwarzenegger,Linda Hamilton,Edward Furlong
4613,Batman Begins,8.2,1648376,2000,"Crime, Drama, Action",Christian Bale,Michael Caine,Ken Watanabe


----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Entrainement du modele
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------